In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

loss_grid = np.load("results/lsh_heatmap_large.npy")

include_values = list(range(300,1200,50)) 
depth_values = list(range(5,23,1))

plt.figure(figsize=(8, 6))
sns.heatmap(loss_grid,
            xticklabels=np.round(include_values, 2),
            yticklabels=np.round(depth_values, 2),
            cmap="viridis",
            cbar_kws={"label": "Average K-Means Loss over 50 Randomised Trials"} )

plt.xlabel("Min Include Threshold")
plt.ylabel("Max Depth")
plt.title("Parameter Heatmap for Large Synthetic Dataset")
plt.show()

In [ ]:
import numpy as np 
from bucket import create_bucket_synopsis, bucket_using_privacy_accountant, Params
from evaluation_utils import kmeans_loss
from lloyd import lloyd_with_weights, dplloyd, PrivacyBudget
from grid import create_grid_synopsis_large

master_rng = np.random.default_rng(42)

def lsh_experiment(algo: int, data: np.ndarray, p: Params, n_trials: int = 20):
    s = master_rng.integers(low=0, high=100000)
    total_loss = 0
    n_successful_trials = n_trials
    for x in range(n_trials):
        if algo == 1:
            print("starting synopsis... ")
            private_points, private_weights = create_bucket_synopsis(data, p, s+x, use_gaussian=True)
        else:
            private_points, private_weights = bucket_using_privacy_accountant(data, p, s+x)
        if private_points.shape[0] <= p.k: # if number of points is less than or equal to desired number of centers
            centers = private_points
        else:
            centers = lloyd_with_weights(k=p.k, X=private_points, weights=private_weights, n_iter=5, rs=s+x)
        try:
            loss = kmeans_loss(centers, data)
        except:
            loss = 0
            n_successful_trials -=1
        total_loss += loss
        print(f"Trial {x+1} done")
    print("Number completed trials: ", n_successful_trials)
    return total_loss / n_successful_trials

large = np.load("datasets/large-synthetic.npy")

results = []

for k in range(1,21):
    p = Params(epsilon=1, delta=1e-6, radius=4.7, dimension=100, k=10, max_depth=15)
    results.append(lsh_experiment(algo=1, data=large, p=p, n_trials=50))